In [40]:
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import r2_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.impute import SimpleImputer
from data_frame_selector import DataFramSelector

import numpy as np
import pandas as pd

In [69]:
def kNN(X_train, X_test, y_train, y_test):
    num_neighbors = 9
    knn = KNeighborsRegressor(num_neighbors, weights='uniform')
    knn.fit(X_train, y_train)
    y_hat = knn.predict(X_test)
    print('KNN R^2 Score ', r2_score(y_test, y_hat))

In [68]:
def svm(X_train, X_test, y_train, y_test):
    svr_rbf = SVR(kernel='rbf', C=1e3, gamma=0.001)
    svr_rbf.fit(X_train, y_train)
    y_hat = svr_rbf.predict(X_test)
    print('SVM R^2 Score ', r2_score(y_test, y_hat))
 
    

In [67]:
def rand_forest(X_train, X_test, y_train, y_test):
    regr = RandomForestRegressor(random_state=0, n_estimators=250)
    regr.fit(X_train, y_train)
    y_hat = regr.predict(X_test)
    print('Random Forest R^2 Score ', r2_score(y_test, y_hat))
    

In [ ]:
def nn(X_train, X_test, y_train, y_test):

In [70]:
df = pd.read_excel('/home/j/Desktop/NHANS/NHANES-6-21-17.xlsx', index=False)

In [71]:
labels = {'DXDTOBMC':'Bone mineral content', 'DXDTOLE':'Total lean excl BMC',
          'DXDTOFAT':'Total fat', 'lbdldl':'LDL - cholesterol',
          'lbdhdl':'HDL - cholesterol', 'lbxtc':'Total cholesterol',
          'lbxglu':'Fasting glucose'}

for label in labels:
    # drop rows with missing y values, i.e. the value to be predicted
    df_processed = df.dropna(subset=[label])
    
    # grab the value to be predicted from dataframe
    y = df_processed[[label]].values.ravel()
    
    # Preprocessing
    
    # columns which have numerical values
    features_numerical = ['bmxwt', 'bmxht','bmxwaist','BMXLEG', 'BMXCALF', 
            'BMXARML', 'BMXARMC', 'BMXTHICR', 'RIDAGEYR']

    # columns which have categorical values
    features_categorical = ['RACETH', 'RIAGENDR']

    # transforming features, adding median value of column to row entry
    # with missing data
    numerical_pipeline = Pipeline([
        ('selector', DataFramSelector(features_numerical)),
        ('imputer', SimpleImputer(missing_values=np.nan, strategy='median')),
        # standardize features
        ('std_scaler', StandardScaler())
    ])

    categorical_pipeline = Pipeline([
        ('selector', DataFramSelector(features_categorical)),
        ('cat_encoder', OneHotEncoder(handle_unknown='ignore'))
    ])

    # full union with categorical and numerical attributes
    full_pipeline = FeatureUnion(transformer_list=[
        ('numerical_pipeline', numerical_pipeline),
        ('categorical_pipeline', categorical_pipeline),
    ])

    X_processed = full_pipeline.fit_transform(df_processed)

    X_train, X_test, y_train, y_test = train_test_split(X_processed, y, test_size=0.2, random_state=0)
    
    print('For', labels[label], ':')
    kNN(X_train, X_test, y_train, y_test)
    svm(X_train, X_test, y_train, y_test)
    rand_forest(X_train, X_test, y_train, y_test)
    print('')

For Bone mineral content :
KNN R^2 Score  0.7999073999512423
SVM R^2 Score  0.80770741932915
Random Forest R^2 Score  0.8313618214404362

For Total lean excl BMC :
KNN R^2 Score  0.9468112705168442
SVM R^2 Score  0.9302824045589024
Random Forest R^2 Score  0.9641589119695609

For Total fat :
KNN R^2 Score  0.9365074533410367
SVM R^2 Score  0.9102672215373173
Random Forest R^2 Score  0.9575327148349525

For LDL - cholesterol :
KNN R^2 Score  0.12310588289463609
SVM R^2 Score  0.1635071696678192
Random Forest R^2 Score  0.17554932401343604

For HDL - cholesterol :
KNN R^2 Score  0.13980140684848408
SVM R^2 Score  0.1857638410799899
Random Forest R^2 Score  0.18446050504673317

For Total cholesterol :
KNN R^2 Score  0.2199940311021682
SVM R^2 Score  0.2650468860874047
Random Forest R^2 Score  0.27469492846180976

For Fasting glucose :
KNN R^2 Score  0.10493691142528905
SVM R^2 Score  0.06800229013695025
Random Forest R^2 Score  0.14760266873875472

